# 02 (trial) — TweakReg Alignment

Diagnostic alignment notebook for quasars with problematic WCS solutions.
Runs TweakReg against reference catalogs with `updatehdr=False` so you can
inspect residuals before committing any WCS changes to the FLT headers.

**Input:** FLT files in `data/raw/<target>/`  
**Reference catalogs:** supply `.cat` files alongside this notebook  
**Output:** shift files written to `notebooks/trial/` for inspection

In [ ]:
import yaml
import numpy as np
from pathlib import Path
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.units import Quantity
from astroquery.gaia import Gaia
from drizzlepac import tweakreg
from IPython.display import Image, display, clear_output

In [ ]:
cfg          = yaml.safe_load(open('../config/wfc3_ir_drizzle_params.yaml'))
tweakreg_cfg = cfg['tweakreg']

RAW_DIR      = Path('../data/raw')
PROC_DIR     = Path('../data/processed')
TWEAKREG_DIR = Path('../data/tweakreg')
TWEAKREG_DIR.mkdir(exist_ok=True)

tweakreg_targets = {}  # {target_name: [flt_path, ...]}

for quasar_dir in sorted(RAW_DIR.iterdir()):
    flt_files = sorted(quasar_dir.glob('*flt.fits'))
    if not flt_files:
        continue

    wcsnames = [fits.getval(str(f), 'WCSNAME', ext=1) for f in flt_files]
    unique = set(wcsnames)

    if len(unique) > 1:
        tweakreg_targets[quasar_dir.name] = [str(f.resolve()) for f in flt_files]
        print(f'NEEDS TWEAKREG: {quasar_dir.name}')
        for name, count in sorted((n, wcsnames.count(n)) for n in unique):
            print(f'  {count:2d} files — {name}')
    else:
        print(f'OK: {quasar_dir.name} — {next(iter(unique))}')

print(f'\n{len(tweakreg_targets)} quasar(s) flagged for TweakReg.')

In [ ]:
target_coords = {}

for target, flt_files in tweakreg_targets.items():
    ra  = fits.getval(flt_files[0], 'RA_TARG',  ext=0)
    dec = fits.getval(flt_files[0], 'DEC_TARG', ext=0)
    target_coords[target] = SkyCoord(ra=ra, dec=dec, unit=(u.deg, u.deg))
    print(f'{target}: RA={ra:.6f}, Dec={dec:.6f}')

In [ ]:
radius_gaia = Quantity(tweakreg_cfg['search_radius'], u.arcmin)

gaia_catalogs = {}  # {target: Table}

for target, flt_files in tweakreg_targets.items():
    coord = target_coords[target]
    print(f'{target}')

    gaia_query = Gaia.query_object_async(coordinate=coord, radius=radius_gaia)
    gaia_catalogs[target] = gaia_query
    print(f'  {len(gaia_query)} Gaia sources within {tweakreg_cfg["search_radius"]:.0f} arcmin')
    print(gaia_query)

In [ ]:
gaia_reduced = {}  # {target: reduced Table}

for target, gaia_query in gaia_catalogs.items():
    reduced_query = gaia_query['ra', 'dec', 'phot_g_mean_mag']
    catfile = str(TWEAKREG_DIR / f'{target}_gaia.cat')
    reduced_query.write(catfile, format='ascii.commented_header', overwrite=True)
    gaia_reduced[target] = reduced_query
    print(f'{target}: {len(reduced_query)} sources → {catfile}')
    print(reduced_query)

In [ ]:
tweakreg_dir_abs = TWEAKREG_DIR.resolve()

for target, flt_files in tweakreg_targets.items():
    print(f'Running TweakReg for {target} ({len(flt_files)} FLTs)...')
    tweakreg.TweakReg(flt_files,
                      imagefindcfg={'threshold': tweakreg_cfg['threshold'],
                                    'conv_width': tweakreg_cfg['conv_width']},
                      refcat=str(tweakreg_dir_abs / f'{target}_gaia.cat'),
                      interactive=False,
                      see2dplot=False,
                      residplot='both',
                      shiftfile=True,
                      outshifts=str(tweakreg_dir_abs / f'{target}_Gaia_shifts.txt'),
                      runfile=str(tweakreg_dir_abs / f'{target}_tweakreg.log'),
                      wcsname=tweakreg_cfg['wcsname'],
                      reusename=True,
                      ylimit=tweakreg_cfg['ylimit'],
                      fitgeometry=tweakreg_cfg['fitgeometry'],
                      minobj=tweakreg_cfg['minobj'],
                      searchrad=tweakreg_cfg['searchrad'],
                      writecat=False,
                      updatehdr=False)
    clear_output()

    for pattern in ('residuals_*.png', 'vector_*.png'):
        for f in Path('.').glob(pattern):
            f.rename(tweakreg_dir_abs / f.name)

    shiftfile = tweakreg_dir_abs / f'{target}_Gaia_shifts.txt'
    if shiftfile.exists():
        lines = [l for l in shiftfile.read_text().splitlines()
                 if l.strip() and not l.startswith('#')]
        fitted   = [l for l in lines if 'nan' not in l]
        unfitted = [l for l in lines if 'nan'     in l]
        print(f'{target}: {len(fitted)}/{len(lines)} images fit  |  {len(unfitted)} failed (nan shifts)')

    rootnames = sorted({p.stem.replace('residuals_', '')
                        for p in tweakreg_dir_abs.glob('residuals_*.png')})
    for root in rootnames:
        resid  = tweakreg_dir_abs / f'residuals_{root}.png'
        vector = tweakreg_dir_abs / f'vector_{root}.png'
        print(f'\n{root}')
        if resid.exists():
            display(Image(filename=str(resid)))
        if vector.exists():
            display(Image(filename=str(vector)))